In [1]:
import string
from tensorflow.keras.utils import to_categorical

PADDING_CHAR = '_'
SPECIAL_SYMBOLS = "-=+@!$%&*()[]{}:;,.?/\\|<>~`^'\""

ALPHANUM_CHARS = string.digits + string.ascii_uppercase + string.ascii_lowercase + PADDING_CHAR + SPECIAL_SYMBOLS
CHAR_TO_INDEX = {char: i for i, char in enumerate(ALPHANUM_CHARS)}
INDEX_TO_CHAR = {i: char for char, i in CHAR_TO_INDEX.items()}
NUM_CLASSES = len(ALPHANUM_CHARS)
MAX_LEN = 9

In [2]:
import os
import cv2
import numpy as np

def load_alphanum_dataset(folder):
    X, y = [], []

    for filename in os.listdir(folder):
        if filename.endswith('.jpg') or filename.endswith('.JPG')  or filename.endswith('.jpeg') or filename.endswith('.png'):
            filepath = os.path.join(folder, filename)
            img = cv2.imread(filepath)
            if img is None:
                continue
            img = cv2.resize(img, (64, 64))
            img = img.astype("float") / 255.0 
            X.append(img)

            label = os.path.splitext(filename)[0]  # Assuming the label is the filename without extension
            label = label.ljust(MAX_LEN, PADDING_CHAR)[:MAX_LEN]  # Pad or truncate to MAX_LEN

            one_hot_label = [to_categorical(CHAR_TO_INDEX[char], num_classes=NUM_CLASSES) for char in label]
            y.append(one_hot_label)

    X = np.array(X)
    y = np.array(y)
    return X, y

In [3]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def create_alphanum_model():
    input_img = Input(shape=(64, 64, 3))
    
    x = Conv2D(32, (3, 3), activation='relu')(input_img)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Flatten()(x)

    outputs = [Dense(NUM_CLASSES, activation='softmax', name=f'char_{i}')(x) for i in range(MAX_LEN)]

    model = Model(inputs=input_img, outputs=outputs)
    
    return model

In [ ]:
X, y = load_alphanum_dataset(r"H:\PROGRAMMING\Bidding_bot\Captcha\Main")
y= np.array(y)
print(y.shape)
y_list = [y[:, i] for i in range(MAX_LEN)]

model = create_alphanum_model()

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy']*MAX_LEN)

model.fit(X, y_list, epochs=5000, batch_size=32, validation_split=0.2)

model.save('captcha_solver_alphanum.h5')

(31, 9, 93)
Epoch 1/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - char_0_accuracy: 0.0000e+00 - char_0_loss: 4.5130 - char_1_accuracy: 0.0000e+00 - char_1_loss: 4.6200 - char_2_accuracy: 0.0000e+00 - char_2_loss: 4.4395 - char_3_accuracy: 0.0000e+00 - char_3_loss: 4.6274 - char_4_accuracy: 0.0000e+00 - char_4_loss: 4.5805 - char_5_accuracy: 0.0000e+00 - char_5_loss: 4.6340 - char_6_accuracy: 0.0000e+00 - char_6_loss: 4.3602 - char_7_accuracy: 0.0000e+00 - char_7_loss: 4.8448 - char_8_accuracy: 0.0000e+00 - char_8_loss: 4.6170 - loss: 41.2364

In [29]:
def decode_prediction(prediction):
    decoded = []
    for char_pred in prediction:
        char_index = np.argmax(char_pred)
        decoded.append(INDEX_TO_CHAR[char_index])
    return ''.join(decoded).strip()

In [ ]:
from PIL import Image
import numpy as np

def preprocess_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img = img.resize((64, 64))  # Resize to match model input
    img_array = np.array(img) / 255.0  # Normalize pixel values
    return np.expand_dims(img_array, axis=0)  # Add batch dimension

X_test = preprocess_image(r"H:\PROGRAMMING\Bidding_bot\Captcha\Main\XKWDN.JPG")

predictions = model.predict(X_test)

predicted_label = ""
for i in range(len(predictions)):
    char_index = np.argmax(predictions[i])
    predicted_label += INDEX_TO_CHAR[char_index]
    
print(f"Predicted label: {predicted_label.strip(PADDING_CHAR)}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Predicted label: XKWD
